cell 0, bootstrap, import libraries and helper functions

In [4]:
%load_ext autoreload
%autoreload 2
# --- Cell 0: Universal Notebook Bootstrap ---
import os, sys
from dotenv import load_dotenv

# Dynamically set project root so imports always work
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Load environment variables (for MySQL, etc.)
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

print(f"✅ Environment ready. Project root: {PROJECT_ROOT}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Environment ready. Project root: d:\dev\cardivore_ai


In [ ]:
import os
import pandas as pd
from cardivore_ai.utils.io_utils import normalize_market_movers_df
from cardivore_ai.utils.db_utils import get_engine_from_env, create_table_if_not_exists

# --- Setup ---
engine = get_engine_from_env()
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")

# --- Read CSVs ---
raw_path = os.path.join(DATA_DIR, "raw_historic_sales.csv")
psa_path = os.path.join(DATA_DIR, "psa10_historic_sales.csv")

raw_df = pd.read_csv(raw_path)
psa_df = pd.read_csv(psa_path)

# --- Normalize / clean using your new universal function ---
raw_df = normalize_market_movers_df(raw_df)
psa_df = normalize_market_movers_df(psa_df)

print("✅ Cleaned numeric fields and standardized columns:")
print(raw_df.head(3))

# --- Create or replace tables in MySQL ---
create_table_if_not_exists(engine, raw_df, "raw_historic_sales")
create_table_if_not_exists(engine, psa_df, "psa10_historic_sales")

raw_df.to_sql("raw_historic_sales", engine, if_exists="replace", index=False)
psa_df.to_sql("psa10_historic_sales", engine, if_exists="replace", index=False)

print("✅ Tables created and loaded: raw_historic_sales, psa10_historic_sales")


In [13]:
# --- Cell: Build and Load Card Summary ---
from cardivore_ai.utils.pipeline_helper import build_card_summary

# Run the full ETL pipeline: extract, transform, load
summary_df, engine = build_card_summary()

# Add search strings & Japanese flag using your improved helper
from cardivore_ai.utils.cards import build_search_string
import pandas as pd

summary_df[["search_string", "is_japanese"]] = summary_df["common_card_name"].apply(
    lambda x: pd.Series(build_search_string(x))
)

# Persist back to MySQL (overwrites or replaces as needed)
summary_df.to_sql("card_summary", engine, if_exists="replace", index=False)

print(f"✅ card_summary table updated. {len(summary_df)} records loaded.")
summary_df.head(10)


✅ card_summary table updated. 25 records loaded.


,common_card_name,raw_avg,raw_volume,raw_last_sale,psa10_avg,psa10_volume,psa10_last_sale,search_string,is_japanese,ebay_url,effective_psa10_value,cost_basis,roi_multiple,roi_percent
0,Acerola's Mischief 2025 Mega Evolution #183/13...,59.60,142.0,48.99,300.00,1.0,300.00,Acerola's Mischief 183/132,False,https://www.ebay.com/sch/i.html?_nkw=Acerola%2...,255.0000,84.60,3.014184,201.418440
1,Bulbasaur 2025 Mega Evolution #133/132 Illustr...,36.86,380.0,29.84,357.00,5.0,410.00,Bulbasaur 133/132,False,https://www.ebay.com/sch/i.html?_nkw=Bulbasaur...,303.4500,61.86,4.905432,390.543162
2,Gumshoos 2025 Mega Evolution #153/132 Illustra...,19.57,523.0,11.88,29.20,6.0,20.00,Gumshoos 153/132,False,https://www.ebay.com/sch/i.html?_nkw=Gumshoos+...,24.8200,44.57,0.556877,-44.312318
3,Ivysaur 2025 Mega Evolution #134/132 Illustrat...,37.81,423.0,31.04,37.50,2.0,35.00,Ivysaur 134/132,False,https://www.ebay.com/sch/i.html?_nkw=Ivysaur+1...,31.8750,62.81,0.507483,-49.251712
4,Lillie's Determination 2025 Mega Evolution #16...,31.60,388.0,27.63,29.97,1.0,29.97,Lillie's Determination 169/132,False,https://www.ebay.com/sch/i.html?_nkw=Lillie%27...,25.4745,56.60,0.450080,-54.992049
5,Lillie's Determination 2025 Mega Evolution #18...,177.88,278.0,160.00,667.86,7.0,800.00,Lillie's Determination 184/132,False,https://www.ebay.com/sch/i.html?_nkw=Lillie%27...,567.6810,202.88,2.798112,179.811218
6,Lt. Surge's Bargain 2025 Mega Evolution #185/1...,40.41,135.0,35.00,355.00,1.0,355.00,Lt. Surge's Bargain 185/132,False,https://www.ebay.com/sch/i.html?_nkw=Lt.+Surge...,301.7500,65.41,4.613209,361.320899
7,Marshadow 2025 Mega Evolution #146/132 Illustr...,67.95,844.0,44.14,446.60,10.0,371.71,Marshadow 146/132,False,https://www.ebay.com/sch/i.html?_nkw=Marshadow...,379.6100,92.95,4.084024,308.402367
8,Mega Absol ex 2025 Mega Evolution #161/132 Ult...,11.83,183.0,12.25,207.50,1.0,207.50,Mega Absol ex 161/132,False,https://www.ebay.com/sch/i.html?_nkw=Mega+Abso...,176.3750,36.83,4.788895,378.889492
9,Mega Absol ex 2025 Mega Evolution #180/132 Spe...,119.54,222.0,102.17,566.67,9.0,477.50,Mega Absol ex 180/132,False,https://www.ebay.com/sch/i.html?_nkw=Mega+Abso...,481.6695,144.54,3.332430,233.243047


In [ ]:
# --- Final Demo Cell: ROI Filter Dashboard ---
from cardivore_ai.utils.pipeline_helper import get_summary_df
from cardivore_ai.utils.fe_helpers import display_roi_filter

summary_df, engine = get_summary_df()
display_roi_filter(summary_df)


✅ Loaded existing summary table from DB.
